<a href="https://colab.research.google.com/github/the-lizardking/ict-trading-bot/blob/main/notebooks/setup/render_env_from_drive_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Render Lean .env File from Google Drive Master Secrets

This notebook decrypts your encrypted master secrets file from Google Drive
and generates a small, lean `.env` file for one runtime profile.

**Your secrets are never printed to screen.**

---

## What this notebook does

| Step | What happens |
|---|---|
| 1 | Connects to your Google Drive |
| 2 | Checks that `master-secrets.sops.yaml` and `age-keys.txt` exist |
| 3 | Installs SOPS and age (if not already installed) |
| 4 | Clones or updates the repo |
| 5 | Runs `render_env_from_master.py` to generate the env file |
| 6 | Shows the output path and variable names (no values) |

---

## Before you start

- [ ] You have run `encrypt_google_drive_master_secrets.ipynb` at least once
- [ ] `My Drive / ICT_Bot_Secrets / master-secrets.sops.yaml` exists
- [ ] `My Drive / ICT_Bot_Secrets / age-keys.txt` exists
- [ ] You have deleted the plaintext `master-secrets.yaml` from Drive

---
## Step 1 — Connect to Google Drive

Run the cell below. A pop-up will appear asking you to sign in to your Google
account and allow Colab to access Drive.

In [1]:
from google.colab import drive

drive.mount('/content/drive')
print('Google Drive connected.')

Mounted at /content/drive
Google Drive connected.


---
## Step 2 — Check that required files exist

This cell verifies that `master-secrets.sops.yaml` and `age-keys.txt` are
present in `My Drive / ICT_Bot_Secrets`.

> If you used a different folder name, change `SECRETS_DIR` in the cell below.

In [2]:
import os, sys

# ── Change only if you used a different folder name in Drive ──────
SECRETS_DIR = '/content/drive/MyDrive/ICT_Bot_Secrets'
# ─────────────────────────────────────────────────────────────────

MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')

errors = []

if not os.path.isdir(SECRETS_DIR):
    errors.append(f'Folder not found: {SECRETS_DIR}')
else:
    print(f'[OK]  Folder found: {SECRETS_DIR}')

if not os.path.isfile(MASTER_FILE):
    errors.append(
        f'Encrypted file not found: {MASTER_FILE}\n'
        '       Run encrypt_google_drive_master_secrets.ipynb first.'
    )
else:
    print(f'[OK]  master-secrets.sops.yaml found ({os.path.getsize(MASTER_FILE)} bytes)')

if not os.path.isfile(AGE_KEY_FILE):
    errors.append(
        f'Age key file not found: {AGE_KEY_FILE}\n'
        '       Run encrypt_google_drive_master_secrets.ipynb first to create this key.'
    )
else:
    print(f'[OK]  age-keys.txt found')

if errors:
    print()
    for e in errors:
        print(f'ERROR: {e}')
    sys.exit(1)

print()
print('All required files are present. Proceed to Step 3.')

[OK]  Folder found: /content/drive/MyDrive/ICT_Bot_Secrets
[OK]  master-secrets.sops.yaml found (14477 bytes)
[OK]  age-keys.txt found

All required files are present. Proceed to Step 3.


---
## Step 3 — Install SOPS and age

These are the free, open-source tools used to decrypt the master secrets file.
If they are already installed from a previous run, this step is skipped automatically.

In [3]:
import subprocess, sys, shutil

AGE_VERSION  = 'v1.1.1'
SOPS_VERSION = 'v3.8.1'

def shell(cmd, check=True):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print('Error:', result.stderr.strip())
        if check:
            print('Installation failed. Please run this cell again.')
            sys.exit(result.returncode)
    return result

# age
if shutil.which('age'):
    print('age already installed:', shell('age --version', check=False).stdout.strip())
else:
    print(f'Installing age {AGE_VERSION} ...')
    shell(f'wget -q https://github.com/FiloSottile/age/releases/download/{AGE_VERSION}/age-{AGE_VERSION}-linux-amd64.tar.gz -O /tmp/age.tar.gz')
    shell('tar -xzf /tmp/age.tar.gz -C /tmp/')
    shell('cp /tmp/age/age /tmp/age/age-keygen /usr/local/bin/')
    shell('chmod +x /usr/local/bin/age /usr/local/bin/age-keygen')
    print('age installed:', shell('age --version', check=False).stdout.strip())

# sops
if shutil.which('sops'):
    print('sops already installed:', shell('sops --version', check=False).stdout.strip())
else:
    print(f'Installing SOPS {SOPS_VERSION} ...')
    shell(f'wget -q https://github.com/getsops/sops/releases/download/{SOPS_VERSION}/sops-{SOPS_VERSION}.linux.amd64 -O /usr/local/bin/sops')
    shell('chmod +x /usr/local/bin/sops')
    print('sops installed:', shell('sops --version', check=False).stdout.strip())

print()
print('Tools ready. Proceed to Step 4.')

Installing age v1.1.1 ...
v1.1.1
age installed: v1.1.1
Installing SOPS v3.8.1 ...
sops 3.8.1
[info] a new version of sops (v3.12.2) is available, you can update by visiting: https://github.com/getsops/sops/releases/tag/v3.12.2
sops installed: sops 3.8.1
[info] a new version of sops (v3.12.2) is available, you can update by visiting: https://github.com/getsops/sops/releases/tag/v3.12.2

Tools ready. Proceed to Step 4.


---
## Step 4 — Clone or update the repo

The render script lives in the repo. This step clones it if it is not already
present, or pulls the latest version if it is.

> Your `GITHUB_PAT` (Personal Access Token) is read from the encrypted master
> secrets file automatically — you do not need to paste it here.

In [4]:
import subprocess, os, sys

REPO_DIR  = '/content/ict-trading-bot'
REPO_URL  = 'https://github.com/the-lizardking/ict-trading-bot.git'

def shell(cmd, check=True, env=None):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.returncode != 0:
        if result.stderr.strip():
            print('Error:', result.stderr.strip())
        if check:
            sys.exit(result.returncode)
    return result

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned. Pulling latest ...')
    shell(f'git -C {REPO_DIR} pull --ff-only origin main')
else:
    print('Cloning repo ...')
    shell(f'git clone --depth 1 {REPO_URL} {REPO_DIR}')

print()
print(f'Repo ready at: {REPO_DIR}')
print('Proceed to Step 5.')

Cloning repo ...

Repo ready at: /content/ict-trading-bot
Proceed to Step 5.


---
## Step 5 — Confirm the rendered output filename

**Operator directive 2026-05-03** — there is one canonical render path. The rendered file goes to `~/ict-trading-bot/.env` on the VM (no profile suffix). The single dry/live toggle in the codebase is per-account `mode: live | dry_run` in `config/accounts.yaml` — applied via `RiskManager.dry_run`. There is no profile-level / process-level / strategy-level dry/live toggle.

If you need an account to NOT trade live yet, set its `mode: dry_run` in `config/accounts.yaml` (or flip it from Telegram via `/accounts dry <account_name>`) — do NOT generate a different env file.

> **What this notebook does:** It renders `.env` locally in this Colab session. It does **not** call Bybit, place orders, run training, or restart any VM. Secrets are never printed.


In [ ]:
# No profile picker — there is only one render path post-2026-05-03.
# Per-account dry/live state lives in config/accounts.yaml `mode` field,
# not at render time. This cell is kept as a hook for any future
# render-time options (e.g. testnet vs mainnet endpoint) without
# breaking the cell numbering downstream notebooks reference.
PROFILE = 'live'
print(f'Profile selected : {PROFILE}')
print('Note: this notebook only renders an .env file. It does not call Bybit, place orders, or restart any VM.')
print('Per-account dry/live mode lives in config/accounts.yaml `mode` field.')
print('Proceed to Step 6.')


---
## Step 6 — Generate the env file

This cell runs `render_env_from_master.py` from the repo.

- **No secret values are printed.**
- The output shows only the file path and the variable names written.
- The generated file is placed inside the repo directory in Colab's `/content`.
- You can then copy it to where your script needs it (e.g., the VM or a Colab env loader).

In [ ]:
import subprocess, os, sys

SECRETS_DIR  = '/content/drive/MyDrive/ICT_Bot_Secrets'
MASTER_FILE  = os.path.join(SECRETS_DIR, 'master-secrets.sops.yaml')
AGE_KEY_FILE = os.path.join(SECRETS_DIR, 'age-keys.txt')
REPO_DIR     = '/content/ict-trading-bot'
SCRIPT       = os.path.join(REPO_DIR, 'scripts', 'render_env_from_master.py')

# Output path — renders directly to .env (the systemd unit's
# EnvironmentFile target). Operator directive 2026-05-03 — there is
# no profile-suffixed file; the dry/live toggle lives in
# config/accounts.yaml `mode`, not in the env filename.
OUT_FILE = os.path.join(REPO_DIR, '.env')

cmd = [
    sys.executable,
    SCRIPT,
    '--master', MASTER_FILE,
    '--age-key-file', AGE_KEY_FILE,
    '--profile', PROFILE,
    '--out', OUT_FILE,
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)

# Print stdout (profile name, output path, variable names — no values)
if result.stdout.strip():
    print(result.stdout.strip())

if result.returncode != 0:
    if result.stderr.strip():
        print(result.stderr.strip())
    sys.exit(result.returncode)

print()
print(f'Env file generated: {OUT_FILE}')


---
## What to do next

The rendered file at `/content/ict-trading-bot/.env` is the canonical env file the trader reads on the VM (`EnvironmentFile=/home/ubuntu/ict-trading-bot/.env`). SCP it directly to that path:

```bash
scp /content/ict-trading-bot/.env ubuntu@<VM_IP>:~/ict-trading-bot/.env
ssh ubuntu@<VM_IP> 'sudo systemctl restart ict-trader-live ict-telegram-bot'
```

Both the trader and the Telegram bot read the same `.env` — restart both units after install or `/accounts_status` will keep showing pre-rotation values (BUG-029).

### Verify the install

After the SCP + restart, run on the VM (or via Telegram `/vm`):

```bash
ssh ubuntu@<VM_IP> "head -10 ~/ict-trading-bot/.env | grep -E 'EXCHANGE|TELEGRAM|BYBIT_API_KEY'"
```

Expect to see your exchange selection, Telegram tokens, and per-account `BYBIT_API_KEY_*` names. There is no longer `MODE`, `DRY_RUN`, or `ALLOW_LIVE_TRADING` in the file — the single dry/live toggle is per-account in `config/accounts.yaml` (`mode: live | dry_run`), applied via `RiskManager.dry_run`.

### Per-account mode

Edit `config/accounts.yaml` to set each account's `mode`:

```yaml
accounts:
  bybit_1:
    ...
    mode: live          # or dry_run
  bybit_2:
    ...
    mode: live          # or dry_run
```

Default = `live` per the Autonomous live-trading rule. From Telegram you can flip at runtime: `/accounts dry <account_name>` or `/accounts live <account_name>`.

### Clean up the Colab session

```python
import os
os.remove(OUT_FILE)
```

Then **Runtime → Disconnect and delete runtime**.
